#  Project: Building a Semantic Search Engine

## Load required libraries

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from sentence_transformers import SentenceTransformer
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core import Document
from qdrant_client import QdrantClient, models

We will use the [Mark It Down library](https://github.com/microsoft/markitdown) to extract the PDF content

```
 !pip install 'markitdown[pdf]
````

In [3]:
from markitdown import MarkItDown

## Check if the file is present

In [4]:
filename = "data/BOCG-15-A-97-1.pdf"

_found = os.path.isfile(filename)

if not _found:
    raise Exception(f"{filename}: file not found")
else:
    print(f"{filename} found.")

data/BOCG-15-A-97-1.pdf found.


## Extract text from PDF

Extract the text contanied in the PDF in a `document` variable

In [5]:
md = MarkItDown(enable_plugins=True)
document = md.convert(filename).text_content

print(f"Document length: {len(document)}")

Document length: 170552


## Initialize the text encoder

In [6]:
MAX_TOKENS = 256

# Initialize the encoder
encoder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Implement Three Chunking Strategies

In [7]:
# strategies = ["fixed", "sentence", "paragraph", "semantic"]
strategies = ["fixed", "sentence", "paragraph"]  # semantic strategy is the worst of 4

def fixed_size_chunks(text, chunk_size=100, overlap=20):
    "Splits text into fixed-size token chunks."
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        if chunk_words:  # Only add non-empty chunks
            chunks.append(' '.join(chunk_words))
    
    return chunks

def sentence_chunks(text, max_sentences=3):
    """Group sentences into chunks"""
    import re
    sentences = re.split(r'[.!?\n]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk_sentences = sentences[i:i + max_sentences]
        if chunk_sentences:
            chunks.append('. '.join(chunk_sentences) + '.')
    
    return chunks

def paragraph_chunks(text):
    """Split by paragraphs or double line breaks"""
    chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
    return chunks if chunks else [text]  # Fallback to full text


_snp = SemanticSplitterNodeParser(
        buffer_size=1,
        breakpoint_percentile_threshold=95,
        embed_model=HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
    )

def semantic_chunks(text):
    document = Document(text=text)
    
    nodes = _snp.get_nodes_from_documents([document])  # Pass list of Document objects
    return [n.text for n in nodes]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Chunk the document

In [8]:
chunks = dict(
    fixed=fixed_size_chunks(document),
    sentence=sentence_chunks(document),
    paragraph=paragraph_chunks(document),
    semantic=semantic_chunks(document)
)

print("Total chunks created:")
count_fixed=len(chunks['fixed'])
count_paragraph=len(chunks['paragraph'])
count_sentence=len(chunks['sentence'])
count_semantic=len(chunks['semantic'])
print(f"\t- Fixed: {count_fixed} chunks")
print(f"\t- Paragraphs: {count_paragraph} chunks")
print(f"\t- Sentences: {count_sentence} chunks")
print(f"\t- Semantic: {count_semantic} chunks")
print()
print(f"{count_fixed + count_paragraph + count_sentence + count_semantic} total chunks.")

Total chunks created:
	- Fixed: 320 chunks
	- Paragraphs: 836 chunks
	- Sentences: 1209 chunks
	- Semantic: 42 chunks

2407 total chunks.


## Create Collections and Process Data

In [9]:
client = QdrantClient(url=os.getenv("QDRANT_URL"))

In [10]:
# Create the collection

collection_name = "day1_semantic_search"

if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

# Create a collection with three named vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config={k: models.VectorParams(size=384, distance=models.Distance.COSINE) for k in strategies},
)

# Index fields for filtering (more on this on day 2)
client.create_payload_index(
    collection_name=collection_name,
    field_name="chunk_strategy",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
client.create_payload_index(
    collection_name=collection_name,
    field_name="chunk_index",
    field_schema=models.PayloadSchemaType.INTEGER,
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

In [11]:
# Process and upload data
points = []
point_id = 0

for strategy_name in strategies:
    for chunk_idx, chunk in enumerate(chunks[strategy_name]):
        points.append(
                    models.PointStruct(
                        id=point_id,
                        vector={strategy_name: encoder.encode(chunk).tolist()},
                        payload={
                            "document_name": filename,
                            "chunk": chunk,
                            "chunk_strategy": strategy_name,
                            "chunk_index": chunk_idx,
                        },
                    )
                )
        point_id += 1

client.upload_points(collection_name=collection_name, points=points)
print(f"Total points: {len(points)}")

Total points: 2365


## Test and Compare

In [12]:
def compare_search_results(query):
    """Compare search results across all chunking strategies"""
    print(f"Query: '{query}'\n")

    for strategy in strategies:
        results = client.query_points(
            collection_name=collection_name,
            query=encoder.encode(query).tolist(),
            using=strategy,
            limit=3,
        )

        print(f"--- {strategy.upper()} CHUNKING ---")
        for i, point in enumerate(results.points, 1):
            print(f"{i}. {point.payload['document_name']} ({point.payload['chunk_index']})")
            print(f"   Score: {point.score:.3f}")
            print(f"   Chunk: {point.payload['chunk']}")
            print("····")
        print()

In [13]:
# Test with domain-specific queries
test_queries = [
    "legislación vigente",
    "casos de uso permitidos",
    "casos de uso prohibidos",
]

for query in test_queries:
    compare_search_results(query)

Query: 'legislación vigente'

--- FIXED CHUNKING ---
1. data/BOCG-15-A-97-1.pdf (154)
   Score: 0.496
   Chunk: Reglamento (UE) 2024/1689, de 13 de junio. Artículo 8. Coordinación de las autoridades de vigilancia del mercado. 1. Las relaciones entre las autoridades de vigilancia del mercado y las autoridades notificantes se regirán por los principios de cooperación, confianza y lealtad institucional. 2. Con el fin de garantizar una actuación uniforme, coordinada y eficaz en el desarrollo de las actuaciones que les atribuye esta ley, las autoridades de vigilancia del mercado se suministrarán e intercambiarán cualquier información operativa que asegure la adecuada coordinación en el ejercicio de sus funciones. 1 - 7 9 - A - 5 1 - G
····
2. data/BOCG-15-A-97-1.pdf (277)
   Score: 0.477
   Chunk: Social, aprobado por el Real Decreto Legislativo 8/2015, de 30 de octubre. Artículo 37. Medidas de carácter provisional en el procedimiento sancionador. 1. Iniciado el procedimiento sancionador, l

## Analyze the results

In [14]:
def analyze_chunking_effectiveness():
    """Analyze which chunking strategy works best for your domain"""

    print("CHUNKING STRATEGY ANALYSIS")
    print("=" * 40)

    # Get chunk statistics for each strategy
    for strategy in strategies:
        # Count chunks per strategy
        results = client.scroll(
            collection_name=collection_name,
            scroll_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="chunk_strategy", match=models.MatchValue(value=strategy)
                    )
                ]
            ),
            limit=100,
        )

        chunks = results[0]
        chunk_sizes = [len(chunk.payload["chunk"]) for chunk in chunks]

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Total chunks: {len(chunks)}")
        print(f"  Avg chunk size: {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")
        print(f"  Size range: {min(chunk_sizes)}-{max(chunk_sizes)} chars")


analyze_chunking_effectiveness()

CHUNKING STRATEGY ANALYSIS

FIXED STRATEGY:
  Total chunks: 100
  Avg chunk size: 633 chars
  Size range: 490-748 chars

SENTENCE STRATEGY:
  Total chunks: 100
  Avg chunk size: 153 chars
  Size range: 8-291 chars

PARAGRAPH STRATEGY:
  Total chunks: 100
  Avg chunk size: 249 chars
  Size range: 1-1672 chars


In [18]:
[ch for ch in chunks["paragraph"] if len(ch) < 100]

['BOLETÍN OFICIAL\nDE LAS CORTES GENERALES',
 'CONGRESO DE LOS DIPUTADOS',
 'XV LEGISLATURA',
 'Serie A:\nPROYECTOS DE LEY',
 '12 de junio de 2026',
 'Núm. 97-1',
 'Pág. 1',
 'PROYECTO DE LEY',
 '121/000096 Proyecto  de  Ley  Orgánica  para  el  buen  uso  y  la',
 'gobernanza de la inteligencia artificial.',
 'La Mesa de la Cámara, en su reunión del día de hoy, ha adoptado el acuerdo que se',
 'indica respecto del asunto de referencia.',
 '(121) Proyecto de ley.',
 'Autor: Gobierno',
 'Proyecto de Ley Orgánica para el buen uso y la gobernanza de la inteligencia artificial.',
 'Acuerdo:',
 'En  ejecución  de  dicho  acuerdo  se  ordena  la  publicación  de  conformidad  con  el',
 'artículo 97 del Reglamento de la Cámara.',
 'Palacio  del  Congreso  de  los  Diputados,  8  de  junio  de  2026.—P.D.  El  Secretario',
 'General del Congreso de los Diputados, Fernando Galindo Elola-Olaso.',
 '1\n-\n7\n9\n-\nA\n-\n5\n1\n-\nG\nC\nO\nB',
 ':\ne\nv\nc',
 'BOLETÍN OFICIAL DE LAS CORTES GENERAL